# CPSC 4770/5770 Spring 2026, HW2, Text Generation and Language Model Decoding

## Please write your name and NetID below

NAME:

NetID:

## Introduction

This assignment focuses on conditional text generation, or natural language generation (NLG). In the simplest form, the task can be formulated as
$$
y ← g(x)
$$
where $x$ is the input text sequence, $y$ is the output text sequence, $g$ is the neural text generation model.
For example, $x$ can be an English sentence, $y$ can be a Chinese sentence, and then $g$ is a machine translation model.
Similarly, $x$ can be a long news article, $y$ can be a short summary, and then $g$ is text summarization model.
In this homework, we will use text summarization as our target task.

In this assignment, you will learn how to use existing models using the Huggingface library to perform the same task.
You will get familiar with the pipeline for text generation tasks, including training, inference, and evaluation.

**Acknowledgement:** This assignment was designed primarily by former TA of the course, Yixin Liu and further adapted to this semester by current course staff.


## Grading Rubric (40 points total)

This part is worth 40 points, distributed as follows:

- **Section 2: Training (5 points)**
  - `compute_loss` implementation (3 points)
  - Correct loss value (2 points)

- **Section 4: Decoding Algorithms (31 points)**
  - Greedy Decoding implementation (5 points)
  - Sampling implementation (5 points)
  - Beam Search implementation (8 points)
  - Hyperparameter Analysis (5 points)
  - Top-p (Nucleus) Sampling implementation (8 points)

- **Reflection Questions (4 points, 2 each)**

**Submission Requirements:**
- Submit completed `.ipynb` file via Gradescope
- Ensure all code cells have been executed and outputs are visible
- Answer all reflection questions in complete sentences
- Delete extra print statements and other output that is not part of the assignment

**IMPORTANT SUBMISSION NOTE**: Please submit your notebook having run all cells and hide long debugging output before submission. If you included print or debugging statements, please remove those before your final run and submission.

**AI policy**: Do not use AI tools (e.g., Anthropic CC, OpenAI Codex, Cursor, ChatGPT, Copilot, Gemini, etc) to generate, complete, or debug your code.
Basic IDE autocomplete that finishes syntax (e.g., closing brackets, completing a for/while/if-else template) is fine, but AI-driven code suggestions that implement logic or functionality are not permitted.
If using Colab, please turn off "Show AI-powered inline completions" in your settings.

----

## 1. Huggingface Transformers and Datasets

In part 1, you have implemented the Transformer model yourself. For this part, we will use the libraries provided by [Huggingface](https://huggingface.co/), including the famous [Transformers](https://huggingface.co/docs/transformers/index) library and the [Datasets](https://huggingface.co/docs/datasets/index) library.
Let's first install them.

In [ ]:
!pip install --upgrade datasets huggingface_hub fsspec
import transformers
import datasets

Let's import the other packages. Please check again to make sure you have the GPU access.

### Using GPU in Colab

PyTorch and other deep learning libraries are much faster using GPU acceleration.

**To enable GPU:**
1. Go to Runtime option on the top left
2. Click "Change runtime type"
3. Select "GPU" for Hardware accelerator
4. Click SAVE button

**Important Notes:**
- Colab free tier provides limited GPU time (typically 12-15 hours per day)
- Once you change runtime type, you'll need to run all cells again
- **Tip:** Develop and debug your code using CPU first, then switch to GPU only for final testing and evaluation
- You can check GPU availability by running the cell below

Let's first verify which device you are using.

In [ ]:
!pip install transformers datasets evaluate rouge_score
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch import Tensor

from typing import Tuple, List

import random
import math
import os
import time
import json
import numpy as np
from collections import Counter

# We'll set the random seeds for deterministic results.
SEED = 1

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.enabled = False
torch.backends.cudnn.deterministic = True

class Placeholder:
    @property
    def DO(self):
        raise NotImplementedError("You haven't yet implemented this part of the assignment yet")

TO = Placeholder()


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Pytorch version is: ", torch.__version__)
print("You are using: ", DEVICE)

### XSum on Huggingface Datasets

We are going to focus on Text Summarization. Text Summarization is the task of converting a long document into a short summary and is a classic task in NLP.
We will be using a famous dataset called [XSum](https://aclanthology.org/D18-1206.pdf), which is one of the most commonly used datasets for abstractive text summarization.

We will use the Huggingface Datasets library to get our dataset.
Check out their [documentation](https://huggingface.co/docs/datasets/quickstart) for more information.


Below we load XSum using HF Datasets. Please note that it may take a few minutes to load and prepocess the dataset.

We will only use the first 100 examples in the test set of XSum for the following parts to save the computation time.
The entire test set in fact contains more than 10k examples and the training set contains more than 200k examples.

https://huggingface.co/datasets/xsum

In [ ]:
import datasets
# get the first 100 data examples in the test split
dataset = datasets.load_dataset("xsum", split="test[:100]")
# check the dataset information
print(dataset.info.features)

### BART on Huggingface Transformers

The model we are going to use is [BART](https://aclanthology.org/2020.acl-main.703.pdf) (Bidirectional and Auto-Regressive
Transformers), which we covered in class and shares the same model architecture you implemented in part 1.

The specific BART checkpoint we are going to use is first *pre-trained* on large corpora and then already *fine-tuned* on XSum training set.

We are going to use HF Transformers to load this fine-tuned BART checkpoint. HF Transformers hosts more than 10k of different model [checkpoints](https://huggingface.co/models), and make them easily accessible through the library.

As you will see below, first we import [BART](https://huggingface.co/docs/transformers/model_doc/bart) from Transformers, and then load the model parameters with a specific model checkpoint [`facebook/bart-large-xsum`](https://huggingface.co/facebook/bart-large-xsum).

For the following sections, you will want to take a deep look into the related documentaions: [doc1](https://huggingface.co/docs/transformers/model_doc/bart#transformers.BartForConditionalGeneration) and [doc2](https://huggingface.co/docs/transformers/model_doc/bart#transformers.BartTokenizer).

In [ ]:
from transformers import BartForConditionalGeneration, BartTokenizer
model_name = "facebook/bart-large-xsum"
tokenizer = BartTokenizer.from_pretrained(model_name)  # load the tokenizer
model = BartForConditionalGeneration.from_pretrained(model_name).to(DEVICE)  # load the model

## Dataloader

### The data pipeline

For training or inferencing any model, we first need to transform the raw textual data into model input. As we saw in part1, the model can work with lists of token ids, and for faster processing we always batch the data examples.
This can be done in pytorch using a "DataLoader"

Let's define a dataloader that transforms the data examples into batches of model input.
While we have provided the implementation for you, please make sure you understand the purpose and logic of the function, which is critical for you to finish the following sections.

In [ ]:
def dataloader(batch_size):
    for i in range(0, len(dataset), batch_size):
        batch = dataset[i:i+batch_size]
        articles = tokenizer(
                batch["document"],
                return_tensors="pt",
                padding="longest",
                add_special_tokens=True,
                truncation=True,
                max_length=512,
            ).to(DEVICE)
        input_ids = articles["input_ids"]
        attention_mask = articles["attention_mask"]
        summaries = tokenizer(
                batch["summary"],
                return_tensors="pt",
                padding="longest",
                add_special_tokens=True,
                truncation=True,
                max_length=128,
            ).to(DEVICE)
        target_input_ids = summaries["input_ids"]
        target_attention_mask = summaries["attention_mask"]
        yield {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "target_input_ids": target_input_ids,
            "target_attention_mask": target_attention_mask,
            "articles": batch["document"],
            "summaries": batch["summary"],
        }

## 2. Training Text Summarization Models

In this section we are going to look into the training process of text summarization models.
The standard training algorithm for text summarization model is Maximum Likelihood Estimation (MLE), and the loss function for this training algorithm is the cross-entropy loss:
$$
L_{CE} = - \sum_{i=1}^l \log p_{model}(\tilde{w}_i|\tilde{w}_{0, ..., i-1}, D),
$$
where $D$ is the input document, $\tilde{w}$ is the *reference* summary (the summary in our training data that we want the model to predict), with length $l+1$ including $\tilde{w}_0$ always set to a special token **BOS** (begin-of-sequence), and $\tilde{w}_i$ is the $i$-th token.
Note that this training algorithm is also called *teacher-forcing*, since the model is required to predict the $i$-th token of the reference summary given the prefix of the reference summary before the $i$-th token.

To train the summarization model, we will take a batch of data examples and compute the loss in the *forward step*, and take the gradient of the model parameters w.r.t. the loss function in the *backward step*, then use an optimization algorithm (e.g., gradient descent) to update the model parameters.
Usually such a training process can take a long time.
Due to the computation resource limitations, for this assigment we will only implement the forward step below.
In the next block, you are going to implement a function `compute_loss`, which computes the cross-entropy loss given a batch of input data points and a summarization model and returns the computed loss.
Please note:
1. The computed loss should be averaged over all tokens.
2. The padding token (PAD) should be ignored in computation.
3. You should not pass in the `labels` argument when you call the forward function of the model.

In [ ]:
def compute_loss(model, batch):
    """
    Args:
        model: the summarization model
        batch: a batch of data from the dataloader

    Return:
        loss: the computed average loss
    """
    # TODO: Implement
    return

Let's test the loss of the pretrained and finetuned BART model! The loss should be exactly 1.6260 (although a little rounding error is allowed.)

In [ ]:
batches = dataloader(batch_size=8)
batch = next(batches) # get one batch of data
model = model.eval()
with torch.no_grad():
    loss = compute_loss(model, batch)
print(loss)

## 3. Text Summarization Evaluation

In this section we will focus on the **evaluation** of text summarization models.

Having trained a summarization model, an important question is **how good the model is?**
To answer this question, we will need to perform the following steps:

1. Using the summarization model to generate the *candidate summaries* on the *test set*. This procedure is also called as **inference**.
For an autoregressive NLG model as BART, at one inference step $t$ it predicts the probabiltiy distribution of the next token based on its *own* previously predicted tokens $\hat{w}_{0, ..., t-1}$ and the input document $D$:
$$
p_t(w_i) = p_{model}(w_i|\hat{w}_{0, ..., t-1}, D),
$$
where $w_i$ stands for a possible token in the vocabulary.
The actual token as time stamp $t$ depends on the specific **decoding algorithm**.
For example, if *greedy decoding* is used, then we select the most probable token according to $p_t$:
$$
\hat{w_t} = \arg \max_{w_i} p_t(w_i).
$$
Later we will explore other decoding algorithms that can yield better results.

2. Having obtained the candidate (system-generated) summaries on the test set, we evalaute the quality of the candidate summaries against the reference summaries provided in the dataset.
This can be achieved by either *human evaluation* or *automatic evaluation* with **automatic evaluation metrics**. The evaluation result for one summarization model is usually a numerical score, which can be the quality score of its generated summaries over the test set.
Then, we can compare the performance of different summarization models by comparing their evaluation scores.

### ROUGE for Automatic Summarization Evaluation

In this part, we are going to evaluate the quality of the summaries using the [ROUGE](https://aclanthology.org/W04-1013.pdf) evaluation metric. Rouge is among the most widely used automatic evaluation metrics for text summarization.

While there are more than 100 [variants](https://aclanthology.org/D15-1013.pdf) of ROUGE, we are going to use its most basic settings: ROUGE-N, which is based on n-grams matching between the reference and candidate summaries.
Concretely, the ROUGE-N recall score is as follows:
$$
\textrm{ROUGE-N}_{R} =\frac{\sum_{m \in \mathcal{M}_n} \mathrm{Count}_{\mathrm{Match}}(m)}{\sum_{m \in \mathcal{M}_n} \mathrm{Count}(m)},
$$
where $\mathcal{M}_n$ is the set of unique $n$-grams in the reference summary, $\mathrm{Count}(m)$ is the number of occurrence of n-gram $m$ in the reference summary, $\mathrm{Count}_{\mathrm{Match}}(m)$ is the number of co-occurrence of n-gram $m$ in both the reference summary and candidate summary.

The ROUGE precision score can be defined by replacing the role of reference and candidate summary. Then we can compute the ROUGE [F1](https://en.wikipedia.org/wiki/F-score) score. Please note that ROUGE score is defined at the summary level, and the model performance can be evaluated by averaging over the ROUGE score of its generated summaries over the entire test set.

We can use Huggingface to calculate the rouge score.


Having implemented ROUGE, we will use it to evaluate the performance of the pre-trained BART on XSum.
To do that, we first let the model generate the candidate summaries (the **inference** step).

In [ ]:
from tqdm import tqdm
# first we predict the summaries and store them in a list
# we also store the actual summaries in another list
# we then compare the predicted summaries to the actual summaries using the ROUGE metric
batch_size = 8
batches = dataloader(batch_size=batch_size)  # get batches
total_batches = len(dataset) // batch_size

model = model.eval()
candidates = []
references = []
with torch.no_grad():
    for (i, batch) in enumerate(tqdm(batches, total=total_batches)):
        summary_ids = model.generate(batch["input_ids"], num_beams=6, max_length=62)  # inference
        summaries = tokenizer.batch_decode(summary_ids, skip_special_tokens=True)  # decoding
        candidates.extend(summaries)
        references.extend(batch["summaries"])

Finally, we use ROUGE to evaluate the quality of the candidate summaries.
We provide a tokenization function that tokenizes the summaries as a pre-processing before ROUGE.

In [ ]:
# evaluate the rouge score
import evaluate

rouge = evaluate.load('rouge')

results = rouge.compute(predictions=candidates, references=references)
print(results)

Here are the reference ROUGE scores for your reference:

ROUGE-1 F1: 44.037
ROUGE-2 F1: 21.686
ROUGE-L F1: 36.991
ROUGELsum F1: 36.990

## 4. Decoding Algorithms

Recall from the previous section that the first step we are going to perform when evalauting NLG models is to use the summarization model to generate the *candidate summaries* on the *test set*. This procedure is also called as **inference**.

For an autoregressive NLG model as BART, at one inference step $t$ it predicts the probabiltiy distribution of the next token based on its *own* previously predicted tokens $\hat{w}_{0, ..., t-1}$ and the input document $D$:
$$
p_t(w_i) = p_{model}(w_i|\hat{w}_{0, ..., t-1}, D),
$$
where $w_i$ stands for a possible token in the vocabulary.
The actual token as time stamp $t$ depends on the specific **decoding algorithm**.

In this section, you are going to implement several different decoding algorithms:

1. Greedy Decoding
2. Sampling
3. Beam Search
4. Top-p (Nucleus) Sampling

**Note:** While most of the Huggingface NLG models have a `.generate()` function supporting different decoding algorithms, you are going to implement the algorithms yourself without using this function.

Before you start the following sections, one thing to keep in mind is *when* should the generation stop.
In our case, it stops when either a pre-defined condition is reached (e.g. a maximum generation length), or a special **end-of-sequence (EOS)** token is predicted as the next token.
You can get the index of this token via `tokenizer.eos_token_id`.
You can also access the BOS token and PAD token in similar manners.

**Note**: in the following sections, we always assume the generated summary starts with a BOS token and ends with a EOS token, and the summary length is counted with both of them.

Before we going to the next sections, let's first define an evaluation function using ROUGE. We will use a reference ROUGE implementation instead for consistent results.

In [ ]:
%pip install rouge_score

In [ ]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rouge3'],
                                  use_stemmer=False, split_summaries=False)

def evaluate_rouge(candidates, references):
    scores = [scorer.score(x, y) for x, y in zip(references, candidates)]
    for n in range(1, 4):
        f1, prec, recall, cnt = 0, 0, 0, 0
        for score in scores:
            f1 += score[f"rouge{n}"].fmeasure
            prec += score[f"rouge{n}"].precision
            recall += score[f"rouge{n}"].recall
            cnt += 1
        f1_avg = f1 / cnt
        prec_avg = prec / cnt
        recall_avg = recall / cnt
        print(f"ROUGE-{n} F1: {f1_avg:.3f}, Precision: {prec_avg:.3f}, Recall: {recall_avg:.3f}")

### Greedy Decoding

When greedy decoding is used, we select the most probable token according to $p_t$:
$$
\hat{w_t} = \arg \max_{w_i} p_t(w_i).
$$
Please implement the algorithm below.

In [ ]:
def greedy_decode(model, input_ids, attention_mask, tokenizer, max_length=62):
    """Greedy decoding
    Args:
        model: the model to use
        input_ids: the input ids, shape [batch_size, seq_len]
        attention_mask: the attention mask, shape [batch_size, seq_len]
        max_length: the maximum length of the generated summary, including the <bos> and <eos> tokens
    Returns:
        the generated summary
    """
    # TODO: Implement
    return

Let's test it.

In [ ]:
batches = dataloader(batch_size=8)  # get batches

candidates = []
references = []
with torch.no_grad():
    for (i, batch) in enumerate(batches):
        print(f"batch {i+1}")
        summaries = greedy_decode(model, batch["input_ids"], batch["attention_mask"], tokenizer)
        candidates.extend(summaries)
        references.extend(batch["summaries"])

evaluate_rouge(candidates, references)

Now you are going to implement the sampling algorithm, which samples the next token according to the probability $p_t(w_i)$.

In [ ]:
def sampling(model, input_ids, attention_mask, tokenizer, max_length=62):
    """Sample decoding
    Args:
        model: the model to use
        input_ids: the input ids, shape [batch_size, seq_len]
        attention_mask: the attention mask, shape [batch_size, seq_len]
        max_length: the maximum length of the generated summary, including the <bos> and <eos> tokens
    Returns:
        the generated summary
    """
    # TODO: Implement
    return

Let's test it.

In [ ]:
batches = dataloader(batch_size=8)  # get batches

candidates = []
references = []
with torch.no_grad():
    for (i, batch) in enumerate(batches):
        print(f"batch {i+1}")
        summaries = sampling(model, batch["input_ids"], batch["attention_mask"], tokenizer)
        candidates.extend(summaries)
        references.extend(batch["summaries"])

evaluate_rouge(candidates, references)

Now you are going to implement the Beam Search algorithm.

One important hyper-parameter of Beam Search is **beam_size**.
Specifically, let **beam_size** to be $n$, Beam Search maintains $n$-possible candidates during search. The candidates are selected by the following scoring function:
$$
f(\hat{w}) = \frac{\sum_{t=1}^{l} \log p_{model}(\hat{w}_t|\hat{w}_{0, ..., t-1}, D)}{l^α},
$$
where $\hat{w}$ is a (partial-)candidate summary generated so far of which the length is $l+1$.
$\alpha$ is another important parameter for *length penalty*.
Specifically, when $\alpha=1$, $f(\hat{w})$ is the average token-level log-probability.
When $\alpha=0$, $f(\hat{w})$ is the sequence-level log-probability.

While there are multiple variants of Beam Search ([reference](https://aclanthology.org/W17-3207.pdf)), we are going to implement its most basic form with three arguments controlling the algorithm behavior: the **beam_size** $n$, the **length penalty factor** $\alpha$, and the maximum summary length.

Please also note that when one of the candidate reaches the EOS token, it means that the search on this candidate is complete. You should maintain this completed candidate until there are at least $n$ other possible candidates with higher scores.

Now you are going to implement Beam Search below:

In [ ]:
def beam_search(model, input_ids, attention_mask, tokenizer, max_length=62, beam_size=4, length_penalty=1.0):
    """Beam search decoding
    Args:
        model: the model to use
        input_ids: the input ids, shape [batch_size, seq_len]
        attention_mask: the attention mask, shape [batch_size, seq_len]
        max_length: the maximum length of the generated summary, including the <bos> and <eos> tokens
        beam_size: the beam size
        length_penalty: the length penalty
    Returns:
        the generated summary
    """
    # TODO: Implement
    return

Let's test it.

In [ ]:
%%time
batches = dataloader(batch_size=8)  # get batches

candidates = []
references = []
with torch.no_grad():
    for (i, batch) in enumerate(batches):
        print(f"batch {i+1}")
        summaries = beam_search(model, batch["input_ids"], batch["attention_mask"], tokenizer,
                                max_length=62, beam_size=6, length_penalty=1.0)
        candidates.extend(summaries)
        references.extend(batch["summaries"])

evaluate_rouge(candidates, references)

Finally, we are going to analyze the behavior of Beam Search w.r.t. the beam size and the length penalty factor.
Now we are going to use the model's builtin `generate` function for better consistency.
Below is a wrapper for this function.

In [ ]:
def beam_search_ref(model, input_ids, attention_mask, tokenizer, max_length=62, beam_size=4, length_penalty=1.0):
    """Beam search decoding
    Args:
        model: the model to use
        input_ids: the input ids, shape [batch_size, seq_len]
        attention_mask: the attention mask, shape [batch_size, seq_len]
        max_length: the maximum length of the generated summary, including the <bos> and <eos> tokens
        beam_size: the beam size
        length_penalty: the length penalty
    Returns:
        the generated summary
    """
    return model.generate(input_ids, num_beams=beam_size, max_length=max_length, length_penalty=length_penalty)

Now please do a hyper-parameter search to find the optimal configuration:

**Search Space:**
- Beam size: integers in range [1, 10] (i.e., 1, 2, 3, ..., 10)
- Length penalty: values in range [0.5, 2.0] with step size 0.1 (i.e., 0.5, 0.6, 0.7, ..., 2.0)

**Instructions:**
1. Perform a grid search over all combinations of the above parameters
2. For each configuration, compute the ROUGE-1 F1 score on the test set
3. Create a 3D scatter plot (eg.https://matplotlib.org/stable/gallery/mplot3d/scatter3d.html) showing:
   - X-axis: beam size
   - Y-axis: length penalty
   - Z-axis: ROUGE-1 F1 score
   - Use different colors/sizes to indicate performance levels
4. Identify at least one configuration that achieves ROUGE-1 F1 ≥ 43.8

In [ ]:
GOOD_HYPER_PARAMETERS = {
    "beam_size": -1,
    "length_penalty": 0,
}   #  TODO:

### Top-p Sampling

In addition to the basic decoding strategies you've implemented, you'll now implement nucleus sampling (also known as top-p sampling). This decoding strategy helps balance between the deterministic nature of greedy/beam search and the potential randomness of pure sampling.
For top-p sampling, at each step we:

- Sort the vocabulary by probability  
- Keep only the minimal set of tokens whose cumulative probability exceeds p  
- Renormalize the probabilities of this reduced set  
- Sample from this renormalized distribution  

In [ ]:
def nucleus_sampling(model, input_ids, attention_mask, tokenizer, max_length=62, p=0.9):
    """Nucleus (top-p) sampling
    Args:
        model: the model to use
        input_ids: the input ids, shape [batch_size, seq_len]
        attention_mask: the attention mask, shape [batch_size, seq_len]
        max_length: the maximum length of the generated summary
        p: the cumulative probability threshold (typically 0.9 or 0.95)
    Returns:
        the generated summary
    """
    # TODO: Implement
    return

In [ ]:
# Test your implementation

batches = dataloader(batch_size=8)  # get batches

candidates = []
references = []
with torch.no_grad():
    for (i, batch) in enumerate(batches):
        print(f"batch {i+1}")
        summaries = nucleus_sampling(model, batch["input_ids"],
                                   batch["attention_mask"], tokenizer, p=0.9)
        candidates.extend(summaries)
        references.extend(batch["summaries"])

evaluate_rouge(candidates, references)

### Reflection questions

1: When implementing the compute_loss function, you used teacher forcing (providing the reference summary tokens during training). What are potential limitations of this training approach? How might this create a mismatch between training and inference?

`TODO: Your answer here`


2: Looking at the BART model architecture used in this assignment, how does it compare to the Transformer you implemented in Part 1? What additional components or modifications does BART introduce and why are they useful for summarization?

`TODO: Your answer here`

In [ ]:
raise NotImplementedError, "Please answer the above questions, then comment out this line."